# Cell 1 — Session configuration

In [ ]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

# Cell 2 — Imports & Glue context

In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sc          = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark       = glueContext.spark_session
job         = Job(glueContext)

print("Glue context ready!")

# Cell 3 — Parameters

In [ ]:
# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET         = "fiap-8mlet-techchallenge-f2-bkt"
S3_PREFIX_RAW     = "b3_daily/raw/"
S3_PREFIX_REFINED = "b3_daily/refined/"

raw_path     = f"s3://{S3_BUCKET}/{S3_PREFIX_RAW}"
refined_path = f"s3://{S3_BUCKET}/{S3_PREFIX_REFINED}"

# ── Glue Catalog ──────────────────────────────────────────────────────────────
GLUE_DATABASE = "b3_stocks"
GLUE_TABLE    = "b3_refined"

# ── Confirm ───────────────────────────────────────────────────────────────────
print(f"RAW     : {raw_path}")
print(f"REFINED : {refined_path}")
print(f"CATALOG : {GLUE_DATABASE}.{GLUE_TABLE}")

# Cell 4 - Select just the last 60 files to process

In [ ]:
import boto3

s3_client = boto3.client("s3")

# List all parquet files in the raw prefix
response = s3_client.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_PREFIX_RAW
)

# Sort by filename date descending and take the 60 most recent
files = sorted(
    [obj for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")],
    key=lambda x: x["Key"],
    reverse=True
)

latest_files = files[:60]
latest_paths = [f"s3://{S3_BUCKET}/{f['Key']}" for f in latest_files]

print(f"Total files found : {len(files)}")
print(f"Files selected    : {len(latest_files)}")

# Cell 5 — Read raw parquet files from S3

In [ ]:
# Read only the selected files
df_raw = spark.read.parquet(*latest_paths)

# If 'date' column is absent, extract it from the filename
if "date" not in df_raw.columns:
    df_raw = df_raw.withColumn(
        "date",
        F.to_date(
            F.regexp_extract(F.input_file_name(), r"b3_(\d{8})\.parquet", 1),
            "yyyyMMdd",
        ),
    )

# Ensure consistent types
df_raw = df_raw.withColumn("date", F.col("date").cast("date"))

print(f"Rows loaded : {df_raw.count():,}")
print(f"Columns     : {df_raw.columns}")
df_raw.show(5, truncate=False)

# Cell 6 — Rename columns
## (Requirement 5-B)

In [ ]:
df = (
    df_raw
    .withColumnRenamed("high", "hi")
    .withColumnRenamed("low",  "lo")
    .withColumnRenamed("open",  "op")
    .withColumnRenamed("close",  "cl")
)

print(f"Columns after rename: {df.columns}")
df.show(3, truncate=False)

# Cell 7 — Window specifications
## 10-day rolling window

In [ ]:
# Per ticker, ordered by date — base for all rolling calculations
w_ticker_date = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
)

# 10-day rolling window (current row + 9 preceding)
w_10 = w_ticker_date.rowsBetween(-9, 0)

print("Window specs ready!")

# Cell 8 — Rolling max & min
## (Requirements 5-A + 5-C)

In [ ]:
df = (
    df
    .withColumn("max_10d", F.max("hi").over(w_10))
    .withColumn("min_10d", F.min("lo").over(w_10))
)

df.select("ticker", "date", "hi", "lo", "max_10d", "min_10d") \
  .orderBy("ticker", "date") \
  .show(20, truncate=False)

# Cell 9 — 20-day EMA of close
## (Requirement 5-C)

In [ ]:
N_EMA = 20
k     = 2.0 / (N_EMA + 1)

# Create lag columns: _lag_0 (today) … _lag_19 (19 days ago)
lag_cols = []
for i in range(N_EMA):
    col_name = f"_lag_{i}"
    df = df.withColumn(col_name, F.lag("cl", i).over(w_ticker_date))
    lag_cols.append((col_name, i))

# Build weighted sum
numerator   = F.lit(0.0)
denominator = F.lit(0.0)

for col_name, i in lag_cols:
    weight      = k * ((1 - k) ** i)
    not_null    = F.col(col_name).isNotNull()
    numerator   = numerator   + F.when(not_null, F.col(col_name) * weight).otherwise(0.0)
    denominator = denominator + F.when(not_null, F.lit(weight)).otherwise(0.0)

df = df.withColumn(
    "ema_20d",
    F.round(numerator / denominator, 4),
)

# Drop temporary lag columns — no longer needed
df = df.drop(*[c for c, _ in lag_cols])

df.select("ticker", "date", "cl", "ema_20d") \
  .orderBy("ticker", "date") \
  .show(10, truncate=False)

# Cell 10 — VWAP + daily record count
## (Requirement 5-A)

In [ ]:
# Per ticker, ordered by date — base for all rolling calculations
w_ticker_date = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
)

# 20-day rolling window (current row + 19 preceding)
w_20 = w_ticker_date.rowsBetween(-19, 0)

df = (
    df
    .withColumn(
        "vwap_20d",
        F.round(
            F.sum(F.col("cl") * F.col("vol")).over(w_20)
            / F.sum("vol").over(w_20),
            4,
        ),
    )
)

df.select("ticker", "date", "cl", "vol", "vwap_20d") \
  .orderBy("ticker", "date") \
  .show(10, truncate=False)

# Cell 11 — Final column selection & preview

In [ ]:
df_refined = df.select(
    "ticker",
    "date",
    F.round("op", 4).alias("op"),
    F.round("hi", 4).alias("hi"),
    F.round("lo", 4).alias("lo"),
    F.round("cl", 4).alias("cl"),
    "vol",
    F.round("max_10d", 4).alias("max_10d"),
    F.round("min_10d", 4).alias("min_10d"),
    F.round("ema_20d", 4).alias("ema_20d"),
    "vwap_20d"
)

print(f"Refined row count : {df_refined.count():,}")
print(f"Refined columns   : {df_refined.columns}")
df_refined.show(10, truncate=False)

# Cell 12 — Write refined parquet to S3
## (Requirement 6)

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(
    df_refined
    .repartition("ticker")
    .write
    .mode("overwrite")
    .partitionBy("ticker")
    .parquet(refined_path)
)

print(f"[OK] Refined data written to {refined_path}")